In [ ]:
from Binoculars.binoculars.detector import Binoculars

In [ ]:
from tqdm import tqdm_notebook as tqdm
bino = Binoculars(
    observer_name_or_path='Qwen/Qwen2.5-7B',
    performer_name_or_path='Qwen/Qwen2.5-7B-Instruct'
)

In [ ]:
# ChatGPT (GPT-4) output when prompted with “Can you write a few sentences about a capybara that is an astrophysicist?"
sample_string = '''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
groundbreaking research in astrophysics.'''

print(bino.compute_score(sample_string))  # 0.75661373
print(bino.predict(sample_string))  # 'Most likely AI-Generated'

In [ ]:
import json
import pandas as pd
fox8 = pd.read_csv("../fox8_scores.csv")
fox8.head()

In [ ]:
from tqdm import tqdm
import torch
import gc

batch_size = 16

scores = []
for start in tqdm(range(0, len(fox8), batch_size), desc="Progress"):
    end = start + batch_size
    batch_text = fox8["text"].iloc[start:end].tolist()
    scores.extend(bino.compute_score(batch_text))

    # Clean cache:
    del batch_text
    torch.cuda.empty_cache()
    gc.collect()

    if len(scores) % 10000 == 0:
      data_sample = fox8.iloc[:len(scores)]
      data_sample["bino_score"] = scores
      data_sample.to_parquet("fox8_bino_scores.parquet")

multisocial["bino_score"] = scores
multisocial.to_parquet("fox8_bino_scores.parquet")

In [ ]:
fox8["bino_score"] = scores
fox8.to_parquet("fox8_bino_scores.parquet")

In [ ]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(fox8.label, fox8.bino_score.fillna(0))
print("fox8_23 ROC AUC:", auc)

In [ ]:
score_names = ["scores_ms", "scores_osmdet", "scores", "scores_with_context", "scores_no_context", "bino_score", "label"]
fox8_agg = fox8.groupby("user_id")[score_names].mean().reset_index()

for score_name in score_names:
    print(score_name)
    auc = roc_auc_score(fox8_agg.label.apply(int), fox8_agg[score_name])
    print("fox8_23 ROC AUC:", auc)